In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import os
from tensorflow.keras import layers, applications

2026-01-04 10:57:07.463930: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767524227.643486      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767524227.694351      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767524228.136197      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767524228.136243      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767524228.136247      55 computation_placer.cc:177] computation placer alr

In [2]:
MODEL1_DIR = '/kaggle/input/densenet121-ensemble/DenseNet121_Ensemble'
preprocess_fn1 = applications.densenet.preprocess_input

MODEL2_DIR = '/kaggle/input/resnet50-ensemble/ResNet50_Ensemble'
preprocess_fn2 = applications.resnet.preprocess_input

MODEL3_DIR = '/kaggle/input/vgg19-ensemble/VGG19_Ensemble'
preprocess_fn3 = applications.vgg19.preprocess_input

MODEL4_DIR = '/kaggle/input/xception-ensemble/Xception_Ensemble'
preprocess_fn4 = applications.xception.preprocess_input

MODEL5_DIR = '/kaggle/input/mobilenet-ensemble/MobileNet_Ensemble'
preprocess_fn5 = applications.mobilenet.preprocess_input

In [3]:
TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [4]:
def tta_augmentation(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    return img

In [5]:
def tta_predict(model, preprocess_fn, ds, tta_rounds=5):
    AUTOTUNE = tf.data.AUTOTUNE
    
    val_ds = ds.map(lambda image, label: (preprocess_fn(image), label)).prefetch(AUTOTUNE)
    preds = [model.predict(val_ds)]

    for _ in range(tta_rounds):
        tta_ds = ds.map(lambda image, label: (tta_augmentation(image), label)).prefetch(AUTOTUNE)
        tta_ds = tta_ds.map(lambda image, label: (preprocess_fn(image), label)).prefetch(AUTOTUNE)
        preds.append(model.predict(tta_ds))

    preds = np.array(preds)
    return np.mean(preds, axis=0)

In [6]:
def get_pred_for_ensembling_model(fold_k, val_ds):
    # model1 = tf.keras.models.load_model(os.path.join(MODEL1_DIR, f'DenseNet121_block_2_fold_{fold_k}.keras'), compile=False)
    model2 = tf.keras.models.load_model(os.path.join(MODEL2_DIR, f'ResNet50_block_1_fold_{fold_k}.keras'), compile=False)
    model3 = tf.keras.models.load_model(os.path.join(MODEL3_DIR, f'VGG19_block_1_fold_{fold_k}.keras'), compile=False)
    model4 = tf.keras.models.load_model(os.path.join(MODEL4_DIR, f'Xception_block_2_fold_{fold_k}.keras'), compile=False)
    model5 = tf.keras.models.load_model(os.path.join(MODEL5_DIR, f'MobileNet_block_1_fold_{fold_k}.keras'), compile=False)

    y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
    # pred1 = tta_predict(model1, preprocess_fn1, val_ds)
    pred2 = tta_predict(model2, preprocess_fn2, val_ds)
    pred3 = tta_predict(model3, preprocess_fn3, val_ds)
    pred4 = tta_predict(model4, preprocess_fn4, val_ds)
    pred5 = tta_predict(model5, preprocess_fn5, val_ds)
    ensemble_pred = (pred2 + pred3 + pred4 + pred5) / 4.0

    return ensemble_pred

In [7]:
DATASET_CACHE = {}
def get_validation_fold(k, train_dir=TRAIN_DIR, IMG_SIZE=(224, 224), BATCH_SIZE=32, SEED=24520152):
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=SEED,
    )
    
    AUTOTUNE = tf.data.AUTOTUNE

    return val_ds

In [8]:
def get_predictions_for_fold(k):
    valid_data = get_validation_fold(k)

    y_true = []
    for _, y in valid_data:
        y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_pred = []

    y_pred.append(get_pred_for_ensembling_model(k, valid_data))
    
    y_pred = np.array(y_pred)
    return y_true, y_pred

In [9]:
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    spec = []

    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        spec.append(TN / (TN + FP + 1e-8))

    return np.array(spec)

In [10]:
def evaluate_fold(k):
    y_true, y_pred = get_predictions_for_fold(k)

    model_results = []
    for i in range(len(y_pred)):
        y_pred_label = np.argmax(y_pred[i], axis=1)
        acc = accuracy_score(y_true, y_pred_label)
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        spec_per_class = macro_specificity(y_true, y_pred_label, 3)
        specificity = np.mean(spec_per_class)

        model_results.append({
            'Accuracy': np.array(acc),
            'Precision': np.array(precision),
            'Recall': np.array(recall),
            'F1-Score': np.array(f1),
            'Specificity': np.array(specificity)
        })
    return np.array(model_results)

In [11]:
num_run = 5
num_fold = 5
num_model = 1
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
all_result = []

for i in range(num_run):
    for j in range(1, 6):
        result = evaluate_fold(j)
        all_result.append(result)

all_result = np.array(all_result)
avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for metric in metric_names:
        values = np.array([
            all_result[k][m][metric] for k in range(len(all_result))
        ])
        avg_metrics[m][metric] = values.mean(axis=0)*100

Found 542 files belonging to 3 classes.


I0000 00:00:1767524241.474206      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1767524258.874592     128 service.cc:152] XLA service 0x7c6a8c003a00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767524258.874637     128 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767524259.684731     128 cuda_dnn.cc:529] Loaded cuDNN version 91002


 4/17 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

I0000 00:00:1767524262.989054     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 350ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 478ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 105ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 103ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 103ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 20s 622ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 282ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step
Found 679 files belonging to 

In [14]:
row_names = ['tta-vgg19-resnet50-xception-mobilenet']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result

,Recall,Specificity,Precision,F1-Score,Accuracy
tta-vgg19-resnet50-xception-mobilenet,93.11,97.03,93.55,93.23,94.12


In [13]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{lrrrrr}
\toprule
 & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
tta-densenet121-resnet50-xception-mobilenet & 93.11 & 97.03 & 93.55 & 93.23 & 94.12 \\
\bottomrule
\end{tabular}
\end{table}

